# CSV metadata customization walkthrough
This notebook provides sample code walkthrough for 'CSV metadata customization' feature, a feautre from Amazon Bedrock Knowledge bases which enhances .csv file processing feature that separates content and metadata. .

For more details on this feature, please read this [blog](https://aws.amazon.com/blogs/machine-learning/knowledge-bases-for-amazon-bedrock-now-supports-advanced-parsing-chunking-and-query-reformulation-giving-greater-control-of-accuracy-in-rag-based-applications/#:~:text=Machine%20Learning%20Blog-,Knowledge%20Bases%20for%20Amazon%20Bedrock%20now%20supports%20advanced%20parsing%2C%20chunking,accuracy%20in%20RAG%20based%20applications).

## 1. Import the needed libraries
First step is to install the pre-requisites packages.

In [ ]:
%pip install --upgrade pip --quiet
%pip install -r ../requirements.txt --no-deps --quiet
%pip install -r ../requirements.txt --upgrade --quiet

In [ ]:
# restart kernel
from IPython.core.display import HTML
HTML("<script>Jupyter.notebook.kernel.restart()</script>")

In [ ]:
import botocore
botocore.__version__

This code is part of the setup and used to :
- Add the parent directory to the python system path
- Imports a custom module (BedrockStructuredKnowledgeBase) from `utils` necessary for later executions

In [ ]:
import sys
import logging
from pathlib import Path
import os
import time
import boto3
import pprint
import json

current_path = Path().resolve()
current_path = current_path.parent

if str(current_path) not in sys.path:
    sys.path.append(str(current_path))

# Print sys.path to verify
print(sys.path)


In [ ]:
#Clients
s3_client = boto3.client('s3')
sts_client = boto3.client('sts')
session = boto3.session.Session()
region =  session.region_name
account_id = sts_client.get_caller_identity()["Account"]
bedrock_agent_client = boto3.client('bedrock-agent')
bedrock_agent_runtime_client = boto3.client('bedrock-agent-runtime') 
logging.basicConfig(format='[%(asctime)s] p%(process)s {%(filename)s:%(lineno)d} %(levelname)s - %(message)s', level=logging.INFO)
logger = logging.getLogger(__name__)
region, account_id

In [ ]:
# Load the shared Knowledge Base created in advanced_chunking_options.ipynb
%store -r kb_id
%store -r bucket_name

try:
    kb_id
    bucket_name
    print(f"Loaded shared KB ID: {kb_id}")
    print(f"Loaded shared S3 bucket: {bucket_name}")
except NameError:
    raise RuntimeError(
        "Shared Knowledge Base not found. Please run "
        "'advanced_chunking_options.ipynb' to completion before running this notebook."
    )


In [ ]:
foundation_model = "us.amazon.nova-2-lite-v1:0"


## 2 - Upload CSV dataset to the shared Knowledge Base

We will upload the `video_games.csv` dataset (and its `.metadata.json` companion file generated below) to the shared Knowledge Base's S3 bucket and trigger an ingestion job on the shared Knowledge Base.


### 2.1 Download csv dataset and upload it to Amazon S3


In [ ]:
!mkdir -p ./csv_data

In [ ]:
!wget https://raw.githubusercontent.com/ali-ce/datasets/master/Most-Expensive-Things/Videogames.csv --no-check-certificate -O ./csv_data/video_games.csv

Let's upload the video games data available on the `csv_data` folder to s3.

In [ ]:
def upload_directory(path, bucket_name):
        for root,dirs,files in os.walk(path):
            for file in files:
                file_to_upload = os.path.join(root,file)
                print(f"uploading file {file_to_upload} to {bucket_name}")
                s3_client.upload_file(file_to_upload,bucket_name,file)

upload_directory("csv_data", bucket_name)

Now we start the ingestion job on the shared Knowledge Base. We look up the data source ID associated with the shared KB, then call `start_ingestion_job` directly via boto3 (the downstream notebook does not hold a `BedrockKnowledgeBase` helper instance). We poll `get_ingestion_job` until completion so retrieval calls below do not race ahead of ingestion.


In [ ]:
# Look up the data source ID associated with the shared KB
ds_response = bedrock_agent_client.list_data_sources(knowledgeBaseId=kb_id)
data_source_id = ds_response['dataSourceSummaries'][0]['dataSourceId']
print(f"Data source ID: {data_source_id}")

# Trigger an ingestion job on the shared KB
time.sleep(30)
ingestion_job = bedrock_agent_client.start_ingestion_job(
    knowledgeBaseId=kb_id,
    dataSourceId=data_source_id
)
ingestion_job_id = ingestion_job['ingestionJob']['ingestionJobId']

# Poll until the ingestion job completes
while True:
    job_status = bedrock_agent_client.get_ingestion_job(
        knowledgeBaseId=kb_id,
        dataSourceId=data_source_id,
        ingestionJobId=ingestion_job_id
    )['ingestionJob']['status']
    print(f"Ingestion job status: {job_status}")
    if job_status in ['COMPLETE', 'FAILED', 'STOPPED']:
        break
    time.sleep(15)

if job_status != 'COMPLETE':
    raise RuntimeError(f"Ingestion job ended with status: {job_status}")


### 2.2 Query the Knowledge Base with Retrieve and Generate API - without metadata

Let's test the knowledge base using the [**retrieve_and_generate**](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/bedrock-agent-runtime/client/retrieve_and_generate.html) API. With this API, Bedrock takes care of retrieving the necessary references from the knowledge base and generating the final answer using a foundation model from Bedrock.

'''
query = "List the video games published by Rockstar Games and released after 2010"
'''

Expected Results: Grand Theft Auto V, L.A. Noire, Max Payne 3


In [ ]:
query = "Provide a list of all video games published by Rockstar Games and released after 2010"

In [ ]:
response = bedrock_agent_runtime_client.retrieve_and_generate(
    input={
        "text": query
    },
    retrieveAndGenerateConfiguration={
        "type": "KNOWLEDGE_BASE",
        "knowledgeBaseConfiguration": {
            'knowledgeBaseId': kb_id,
            "modelArn": foundation_model,
            "retrievalConfiguration": {
                "vectorSearchConfiguration": {
                    "numberOfResults":5
                } 
            }
        }
    }
)

pprint.pp(response['output']['text'])


#### 2.3 Prepeare metadata for ingestion


In [ ]:
import csv
import json

In [ ]:
def generate_json_metadata(csv_file, content_field, metadata_fields, excluded_fields):
    # Open the CSV file and read its contents
    with open(csv_file, 'r') as file:
        reader = csv.DictReader(file)
        headers = reader.fieldnames

    # Create the JSON structure
    json_data = {
        "metadataAttributes": {},
        "documentStructureConfiguration": {
            "type": "RECORD_BASED_STRUCTURE_METADATA",
            "recordBasedStructureMetadata": {
                "contentFields": [
                    {
                        "fieldName": content_field
                    }
                ],
                "metadataFieldsSpecification": {
                    "fieldsToInclude": [],
                    "fieldsToExclude": []
                }
            }
        }
    }

    # Add metadata fields to include
    for field in metadata_fields:
        json_data["documentStructureConfiguration"]["recordBasedStructureMetadata"]["metadataFieldsSpecification"]["fieldsToInclude"].append(
            {
                "fieldName": field
            }
        )

    # Add fields to exclude (all fields not in content_field or metadata_fields)
    if not excluded_fields:
        excluded_fields = set(headers) - set([content_field] + metadata_fields)
    
    for field in excluded_fields:
        json_data["documentStructureConfiguration"]["recordBasedStructureMetadata"]["metadataFieldsSpecification"]["fieldsToExclude"].append(
            {
                "fieldName": field
            }
        )

    # Generate the output JSON file name
    output_file = f"{csv_file.split('.')[0]}.csv.metadata.json"

    # Write the JSON data to the output file
    with open(output_file, 'w') as file:
        json.dump(json_data, file, indent=4)

    print(f"JSON metadata file '{output_file}' has been generated.")

In [ ]:
csv_file = 'csv_data/video_games.csv'
content_field = 'Videogame'
metadata_fields = ['Year', 'Developer', 'Publisher']
excluded_fields =['Description']

generate_json_metadata(csv_file, content_field, metadata_fields, excluded_fields)

In [ ]:
# upload metadata file to S3
upload_directory("csv_data", bucket_name)

# delete metadata file from local
os.remove('csv_data/video_games.csv.metadata.json')

Now start the ingestion job. Since, we are using the same documents as used for fixed chunking, we are skipping the step to upload documents to s3 bucket. 

In [ ]:
# Trigger a second ingestion job on the shared KB to pick up the new metadata file
time.sleep(30)
ingestion_job = bedrock_agent_client.start_ingestion_job(
    knowledgeBaseId=kb_id,
    dataSourceId=data_source_id
)
ingestion_job_id = ingestion_job['ingestionJob']['ingestionJobId']

while True:
    job_status = bedrock_agent_client.get_ingestion_job(
        knowledgeBaseId=kb_id,
        dataSourceId=data_source_id,
        ingestionJobId=ingestion_job_id
    )['ingestionJob']['status']
    print(f"Ingestion job status: {job_status}")
    if job_status in ['COMPLETE', 'FAILED', 'STOPPED']:
        break
    time.sleep(15)

if job_status != 'COMPLETE':
    raise RuntimeError(f"Ingestion job ended with status: {job_status}")


### 2.4 Query the Knowledge Base with Retrieve and Generate API - without metadata

create the filter 

In [ ]:
one_group_filter= {
    "andAll": [
        {
            "equals": {
                "key": "Publisher",
                "value": "Rockstar Games"
            }
        },
        {
            "greaterThan": {
                "key": "Year",
                "value": 2010
            }
        }
    ]
}

Pass the filter to `retrievalConfiguration` of the [**retrieve_and_generate**](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/bedrock-agent-runtime/client/retrieve_and_generate.html).

In [ ]:
response = bedrock_agent_runtime_client.retrieve_and_generate(
    input={
        "text": query
    },
    retrieveAndGenerateConfiguration={
        "type": "KNOWLEDGE_BASE",
        "knowledgeBaseConfiguration": {
            'knowledgeBaseId': kb_id,
            "modelArn": foundation_model,
            "retrievalConfiguration": {
                "vectorSearchConfiguration": {
                    "numberOfResults":5,
                    "filter": one_group_filter
                } 
            }
        }
    }
)

print(response['output']['text'])


As you can see, with the retrieve and generate API we get the final response directly, now let's observe the citations for `RetreiveAndGenerate` API. Also, let's  observe the retrieved chunks and citations returned by the model while generating the response. When we provide the relevant context to the foundation model alongwith the query, it will most likely generate the high quality response. 

In [ ]:
response_standard = response['citations'][0]['retrievedReferences']
print("# of citations or chunks used to generate the response: ", len(response_standard))
def citations_rag_print(response_ret):
#structure 'retrievalResults': list of contents. Each list has content, location, score, metadata
    for num,chunk in enumerate(response_ret,1):
        print(f'Chunk {num}: ',chunk['content']['text'],end='\n'*2)
        print(f'Chunk {num} Location: ',chunk['location'],end='\n'*2)
        print(f'Chunk {num} Metadata: ',chunk['metadata'],end='\n'*2)

citations_rag_print(response_standard)